In [1]:
from src.vanna_connector import initialize_vanna
from src.config import QDRANT_URL, QDRANT_API_KEY, OPENAI_API_KEY, OPENAI_MODEL,\
        OPENAI_API_URL, DENSE_EMBEDDING_MODEL_PATH, INTERIM_DATA_DIR, PROCESSED_DATA_DIR, DEVICE
from tqdm import tqdm
import os

/home/user/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Initialize Vanna client

In [ ]:
# initialize configs for the database, vector store and chat model

PATH_TO_DB = os.path.join(PROCESSED_DATA_DIR, "bank_transaction_monitoring", "bank_transaction_monitoring_inline_short.sqlite.db")

# postgress example, for concrete params for different databases check VannaBase class methods connect_to_*
# postgres_config = {
#     "params": {
#         "host": "localhost",
#         "port": 5432,
#         "database": "bank_transaction_monitoring",
#         "user": "postgres",
#         "password": "postgres"
#     },
#     "type": "postgres"}


sqlite_config = {
    "params": {
        "url": str(PATH_TO_DB)
    },
    "type": "sqlite"}


qdrant_config = {"fastembed_model": DENSE_EMBEDDING_MODEL_PATH,
                 "url": QDRANT_URL, 
                 "api_key": QDRANT_API_KEY,
                 "device": DEVICE}

openai_config = {"api_key": OPENAI_API_KEY,
                 "model": OPENAI_MODEL,
                 "base_url": OPENAI_API_URL,
                 "temperature": 0.01}

In [3]:
vanna_client = initialize_vanna(db_config=sqlite_config,
                                qdrant_config=qdrant_config,
                                openai_config=openai_config)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2667.83it/s]
/home/user/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/vanna/legacy/qdrant/qdrant.py:49: UserWarning: Api key is used with an insecure connection.
  self._client = QdrantClient(


# Search functionality
Allows to find top similar SQL-scripts with descriptions to the provided query

In [4]:
query = "Сколько в среднем в месяц тратится на покупки?"
vanna_client.get_similar_question_sql(query, n_results=5)


[{'question': 'Скрипт для формирования списка клиентов из штата CA с расчетом за январь 2020 года количества и общей суммы расходных операций по их активным счетам, с отдельным выделением суммы снятий через ATM и отбором только тех клиентов, у которых совокупные расходы превысили 10000.',
  'sql': "SELECT\n    c.c01 AS client_id,\n    c.c02 AS client_name,\n    COUNT(*) AS expense_txn_count,\n    SUM(-t.t02) AS total_expense_amount,\n    SUM(CASE WHEN t.t03 LIKE 'ATM%' THEN -t.t02 ELSE 0 END) AS atm_withdrawal_amount\nFROM btm_cst AS c\nJOIN btm_accd AS a\n    ON a.a01 = c.c01\nJOIN btm_trn AS t\n    ON t.t01 = a.a02\nWHERE c.c04 = 'CA'\n  AND a.a05 = 'ACTIVE'\n  AND t.t05 >= '2020-01-01'\n  AND t.t05 < '2020-02-01'\n  AND t.t02 < 0\nGROUP BY\n    c.c01,\n    c.c02\nHAVING SUM(-t.t02) > 10000;",
  'id': 'e14e56e5-e939-574f-ab29-ef25fd65841e'},
 {'question': 'Скрипт для выявления клиентов с аномально высоким исходящим оборотом по активным связанным счетам за 2020 год: он сравнивает сумм

# SQL generation functionality
Allows to generate SQL for the given query (uses vector store internally to retrieve relevant DDLs, scripts and documentation).

In [8]:
query = "Сколько в среднем в месяц тратится на покупки?"
sql_script = vanna_client.generate_sql(query, allow_llm_to_see_data=True) # allow_llm_to_see_data=True - allows to make a select request to the connected database to generate SQL

print(sql_script)

SQL Prompt: [{'role': 'system', 'content': "You are a SQLite expert. Please help to generate a SQL query to answer the question. Your response should ONLY be based on the given context and follow the response guidelines and format instructions. \n===Tables \nCREATE TABLE cus ( -- Клиенты проката.\n  h01 INT NOT NULL, -- Идентификатор клиента.\n  h02 INT NOT NULL, -- Идентификатор магазина.\n  h03 VARCHAR(45) NOT NULL, -- Имя клиента.\n  h04 VARCHAR(45) NOT NULL, -- Фамилия клиента.\n  h05 VARCHAR(50) DEFAULT NULL, -- Email клиента.\n  h06 INT NOT NULL, -- Идентификатор адреса.\n  h07 CHAR(1) DEFAULT 'Y' NOT NULL, -- Статус активности.\n  h08 TIMESTAMP NOT NULL, -- Дата регистрации.\n  h09 TIMESTAMP NOT NULL, -- Дата изменения записи.\n  PRIMARY KEY (h01),\n  CONSTRAINT fk_cus_sto FOREIGN KEY (h02) REFERENCES sto (j01) ON DELETE NO ACTION ON UPDATE CASCADE,\n  CONSTRAINT fk_cus_adr FOREIGN KEY (h06) REFERENCES adr (e01) ON DELETE NO ACTION ON UPDATE CASCADE\n);\n\nCREATE TABLE pay ( -- 

In [9]:
vanna_client.run_sql(sql_script) # execute generated SQL in the connected database

,avg_monthly_purchase_spend
0,42166.666667
